# Evidex: entrenar el detector de ediciones con IA

Antes de empezar, en el panel derecho (**Settings**):
1. **Accelerator:** GPU T4 x2 (o P100).
2. **Internet:** activado (para bajar TGIF; Kaggle pide verificar su teléfono).

**Vuelta 3 (mosaico a tamaño real):** el modelo ya no achica la foto entera; la mira en pedazos de 512 px sin
reducirla, para no perder ediciones pequeñas. Para partir desde el modelo anterior (recomendado, aprende más rápido), suba `evidex_modelo.zip` como *Dataset* (Create → New Dataset)
y agréguelo con **+ Add Input**. Si no, parte de cero. Use **Save Version → Save & Run All (Commit)**: corre en segundo
plano y para sola antes de las 9 horas.

Después: **Run all**. Tarda unas 6 a 9 horas. Al final descargue `evidex_modelo.zip` desde la pestaña
**Output** (a la derecha) y copie los 3 archivos a `veritas\modelos\` en su PC.

Datos: TGIF (IDLab, imec), CC BY 4.0, https://github.com/IDLabMedia/tgif-dataset. Se excluyen las variantes FLUX
(licencia no comercial). Fotos propias (opcional): suba un *Dataset* de Kaggle con las carpetas `originales/`,
`editadas/` y opcionalmente `mascaras/`, y agréguelo al notebook con **+ Add Input**. Solo fotos suyas o con permiso.

In [ ]:
# 1. Configuración (puede cambiar estos números)
import os
# Vuelta 3: recortes a tamaño real (la foto ya no se achica a 512) y análisis en mosaico, como en Evidex.
# La vuelta 1 usó 2.000 por carpeta, orig+sd2-sp+sdxl-fr, 15 épocas y la foto entera achicada.
N_POR_CARPETA = int(os.environ.get("EVX_N", 3000))     # imágenes editadas de TGIF por carpeta
N_ORIG = int(os.environ.get("EVX_N_ORIG", 6000))       # originales (más, para no tener 4 editadas por cada original)
CARPETAS = os.environ.get("EVX_CARPETAS", "sd2-sp,sd2-fr,sdxl-fr,ps-sp")
EPOCAS = int(os.environ.get("EVX_EPOCAS", 25))
MAX_HORAS = float(os.environ.get("EVX_MAX_HORAS", 9))  # para antes y exporta lo mejor (Kaggle corta a las 12 h)
LADO = int(os.environ.get("EVX_LADO", 512))
LOTE = int(os.environ.get("EVX_LOTE", 16))
N_PRUEBA = int(os.environ.get("EVX_N_PRUEBA", 300))   # fotos de la partición «testing» para medir al final
PREENTRENADO = os.environ.get("EVX_PREENTRENADO", "1") == "1"   # pesos ImageNet de timm (ver licencias)
DATOS = os.environ.get("EVX_DATOS", "/kaggle/temp/tgif" if os.path.isdir("/kaggle") else "datos_tgif")
SALIDA = os.environ.get("EVX_SALIDA", "/kaggle/working/modelos" if os.path.isdir("/kaggle") else "modelos")

In [ ]:
# 2. Preparar el entorno
import subprocess, sys, shutil
def sh(cmd):
    print("$", cmd, flush=True)
    r = subprocess.run(cmd, shell=True)
    if r.returncode:
        raise SystemExit(f"Falló: {cmd}")
sh(f"{sys.executable} -m pip install -q timm safetensors onnx onnxruntime")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO HAY GPU (active el acelerador en Settings)")
print(shutil.disk_usage("/kaggle" if os.path.isdir("/kaggle") else "."))

In [ ]:
# 3. Código de entrenamiento de Evidex (copiado de training/ del repositorio)
from pathlib import Path
Path("training").mkdir(exist_ok=True)
ARCHIVOS = {
 "datos.py": "\"\"\"Datos para entrenar el detector de ediciones con IA.\n\nTodo pasa por un manifiesto CSV (una fila por imagen), para saber siempre de dónde salió cada foto y con qué\nlicencia:  ruta,etiqueta,mascara,conjunto,licencia\n  etiqueta: 1 = editada/generada con IA, 0 = original\n  mascara: PNG blanco donde se cambió (vacío = sin máscara; para originales se usa todo negro)\n\nConjuntos soportados de entrada:\n  - TGIF / TGIF2 (carpetas orig/, sd2-sp/, sdxl-fr/, ... y masks/), CC BY 4.0 / CC BY-SA 4.0.\n  - Fotos propias (carpeta con originales/ y editadas/, y opcionalmente mascaras/ con el mismo nombre).\n    Use solo fotos suyas o con permiso escrito; los datos reales de BCI requieren autorización de BCI.\n\nLas licencias NO comerciales quedan bloqueadas: el script se niega a usarlas (ver LICENCIAS_BLOQUEADAS).\n\"\"\"\nfrom __future__ import annotations\n\nimport csv\nimport io\nimport random\nimport re\nfrom pathlib import Path\n\nimport numpy as np\nfrom PIL import Image\n\nIMAGES = {\".jpg\", \".jpeg\", \".png\", \".webp\"}\nLICENCIAS_BLOQUEADAS = (\"nc\", \"non-commercial\", \"noncommercial\", \"no comercial\", \"research only\", \"desconocida\")\nCAMPOS = [\"ruta\", \"etiqueta\", \"mascara\", \"conjunto\", \"licencia\"]\n\n# subcarpeta de TGIF -> licencia (README de IDLabMedia/tgif-dataset)\nTGIF_LICENCIA = \"CC BY 4.0 (TGIF) / CC BY-SA 4.0 (TGIF2); imágenes base MS-COCO\"\nTGIF_EVITAR = (\"flux\",)          # FLUX.1 dev tiene licencia no comercial: esas imágenes no se usan\n\n\ndef licencia_permitida(lic: str) -> bool:\n    l = (lic or \"desconocida\").lower()\n    return not any(b in l for b in LICENCIAS_BLOQUEADAS)\n\n\ndef _mascara_tgif(img: Path, masks: Path) -> str:\n    # «368961_mask_bbox.png_ps_mask.png_sd2_0.png» -> masks/<split>/<clase>/368961_mask_bbox.png\n    m = re.match(r\"(.+?_mask_(?:bbox|segm)(?:_512)?\\.png)\", img.name)\n    if not m:\n        return \"\"\n    rel = img.parent.relative_to(img.parents[2])           # <split>/<clase>\n    cand = masks / rel / m.group(1)\n    return str(cand) if cand.exists() else \"\"\n\n\ndef manifiesto_tgif(raiz: Path) -> list[dict]:\n    \"\"\"raiz/orig/<split>/<clase>/*.png, raiz/<metodo>/<split>/<clase>/*.png, raiz/masks/...\"\"\"\n    filas, masks = [], raiz / \"masks\"\n    for carpeta in sorted(p for p in raiz.iterdir() if p.is_dir() and p.name != \"masks\"):\n        if any(x in carpeta.name.lower() for x in TGIF_EVITAR):\n            continue\n        original = carpeta.name == \"orig\"\n        # «-fr» = toda la imagen pasó por la IA (regenerada): la máscara es la imagen completa, no solo la zona\n        completa = carpeta.name.lower().endswith(\"-fr\")\n        for img in sorted(p for p in carpeta.rglob(\"*\") if p.suffix.lower() in IMAGES):\n            mask = \"\" if original or completa else _mascara_tgif(img, masks)\n            if not original and not completa and not mask:\n                continue                                     # editada sin máscara: no sirve para localizar\n            filas.append({\"ruta\": str(img), \"etiqueta\": 0 if original else 1, \"mascara\": mask,\n                          \"conjunto\": f\"TGIF/{carpeta.name}\", \"licencia\": TGIF_LICENCIA})\n    return filas\n\n\ndef manifiesto_propias(raiz: Path, licencia: str = \"propia (fotos de Zelekpress con permiso)\") -> list[dict]:\n    filas = []\n    for sub, et in ((\"originales\", 0), (\"editadas\", 1)):\n        for img in sorted(p for p in (raiz / sub).rglob(\"*\") if p.suffix.lower() in IMAGES):\n            mask = raiz / \"mascaras\" / (img.stem + \".png\")\n            filas.append({\"ruta\": str(img), \"etiqueta\": et, \"mascara\": str(mask) if mask.exists() else \"\",\n                          \"conjunto\": f\"propias/{sub}\", \"licencia\": licencia})\n    return filas\n\n\ndef guardar(filas: list[dict], destino: Path) -> None:\n    malas = sorted({f[\"licencia\"] for f in filas if not licencia_permitida(f[\"licencia\"])})\n    if malas:\n        raise SystemExit(f\"Licencias no permitidas para uso comercial: {malas}\")\n    with destino.open(\"w\", newline=\"\", encoding=\"utf-8\") as fh:\n        w = csv.DictWriter(fh, fieldnames=CAMPOS)\n        w.writeheader()\n        w.writerows(filas)\n\n\ndef leer(manifiesto: Path) -> list[dict]:\n    with manifiesto.open(encoding=\"utf-8\") as fh:\n        filas = list(csv.DictReader(fh))\n    for f in filas:\n        if not licencia_permitida(f[\"licencia\"]):\n            raise SystemExit(f\"{f['ruta']}: licencia no permitida ({f['licencia']})\")\n        f[\"etiqueta\"] = int(f[\"etiqueta\"])\n    return filas\n\n\n# ---- lo que le pasa a una foto en el camino (WhatsApp, capturas, reenvíos) -------------------------------\ndef como_whatsapp(img: Image.Image, mask: Image.Image, rng: random.Random) -> tuple[Image.Image, Image.Image]:\n    \"\"\"Reducción + recompresión JPEG (a veces doble) + recorte leve. Se aplica IGUAL a originales y editadas,\n    para que el modelo no aprenda «comprimida = editada».\"\"\"\n    if rng.random() < .3:                                   # recorte leve, como al reencuadrar en la galería\n        w, h = img.size\n        dx, dy = int(w * rng.uniform(0, .08)), int(h * rng.uniform(0, .08))\n        box = (dx, dy, w - int(w * rng.uniform(0, .08)), h - int(h * rng.uniform(0, .08)))\n        img, mask = img.crop(box), mask.crop(box)\n    lado = rng.choice([640, 800, 1024, 1280, 1600, 2048])\n    if max(img.size) > lado:\n        f = lado / max(img.size)\n        nuevo = (max(1, round(img.width * f)), max(1, round(img.height * f)))\n        img, mask = img.resize(nuevo, Image.LANCZOS), mask.resize(nuevo, Image.NEAREST)\n    for _ in range(rng.choice([1, 1, 2])):                  # a veces se reenvía: doble compresión\n        buf = io.BytesIO()\n        img.save(buf, \"JPEG\", quality=rng.randint(50, 92))\n        img = Image.open(io.BytesIO(buf.getvalue())).convert(\"RGB\")\n    return img, mask\n\n\ndef a_tensor(img: Image.Image, lado: int) -> np.ndarray:\n    \"\"\"Igual que en evidex/forensics/learned.py: RGB, lado x lado, 0..1, normalizado ImageNet, CHW.\"\"\"\n    x = np.asarray(img.convert(\"RGB\").resize((lado, lado), Image.BILINEAR), dtype=np.float32) / 255.0\n    x = (x - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)\n    return x.transpose(2, 0, 1)\n\n\n# ---- mosaico: la foto se analiza en pedazos a tamaño real, sin achicarla a 512 ----------------------------\n# (la misma lógica está en evidex/forensics/learned.py; tests/test_learned_model.py revisa que den lo mismo)\nMAX_LADO = 2048          # fotos más grandes se reducen a esto antes del mosaico (tiempo de análisis acotado)\n\n\ndef posiciones(n: int, lado: int, paso: int) -> list[int]:\n    if n <= lado:\n        return [0]\n    pos = list(range(0, n - lado, paso))\n    return pos + [n - lado]\n\n\ndef preparar(img: Image.Image, max_lado: int = MAX_LADO) -> Image.Image:\n    img = img.convert(\"RGB\")\n    if max(img.size) > max_lado:\n        f = max_lado / max(img.size)\n        img = img.resize((max(1, round(img.width * f)), max(1, round(img.height * f))), Image.LANCZOS)\n    return img\n\n\ndef normalizar(img: Image.Image) -> np.ndarray:\n    x = np.asarray(img, dtype=np.float32) / 255.0\n    return ((x - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32))\n\n\ndef mosaico(img: Image.Image, lado: int, paso: int):\n    \"\"\"Pedazos lado x lado (CHW normalizados) y su posición. Si la foto es más chica que `lado`, se rellena\n    con negro (igual al entrenar y al analizar).\"\"\"\n    w, h = img.size\n    W, H = max(w, lado), max(h, lado)\n    x = np.zeros((H, W, 3), np.float32)\n    x[:] = normalizar(Image.new(\"RGB\", (1, 1)))[0, 0]          # negro normalizado\n    x[:h, :w] = normalizar(img)\n    tiles, pos = [], []\n    for yy in posiciones(H, lado, paso):\n        for xx in posiciones(W, lado, paso):\n            tiles.append(x[yy:yy + lado, xx:xx + lado].transpose(2, 0, 1))\n            pos.append((xx, yy))\n    return np.stack(tiles).astype(np.float32), pos, (w, h), (W, H)\n\n\ndef puntuar_mosaico(correr, img: Image.Image, lado: int, paso: int, lote: int = 8):\n    \"\"\"`correr(lote_np) -> (mapas N x1xh xw, puntajes N)`. Devuelve (puntaje de la foto, mapa de la foto\n    a 1/escala, escala). Puntaje = el del pedazo más sospechoso.\"\"\"\n    tiles, pos, (w, h), (W, H) = mosaico(img, lado, paso)\n    mapas, puntajes = [], []\n    for i in range(0, len(tiles), lote):\n        m, s = correr(tiles[i:i + lote])\n        mapas.append(np.asarray(m)); puntajes.append(np.asarray(s).reshape(-1))\n    mapas, puntajes = np.concatenate(mapas)[:, 0], np.concatenate(puntajes)\n    esc = lado // mapas.shape[-1]\n    acc = np.zeros((H // esc + 1, W // esc + 1), np.float32)\n    cnt = np.zeros_like(acc)\n    for (xx, yy), m in zip(pos, mapas):\n        acc[yy // esc:yy // esc + m.shape[0], xx // esc:xx // esc + m.shape[1]] += m\n        cnt[yy // esc:yy // esc + m.shape[0], xx // esc:xx // esc + m.shape[1]] += 1\n    mapa = (acc / np.maximum(cnt, 1))[:max(1, h // esc), :max(1, w // esc)]\n    return float(puntajes.max()), mapa, esc\n\n\ndef degradar_fijo(img: Image.Image) -> Image.Image:\n    \"\"\"Lo mismo que el banco de pruebas: lado mayor 1600, JPEG 70 (para validar siempre igual).\"\"\"\n    img = img.convert(\"RGB\")\n    img.thumbnail((1600, 1600), Image.LANCZOS)\n    buf = io.BytesIO()\n    img.save(buf, \"JPEG\", quality=70)\n    return Image.open(io.BytesIO(buf.getvalue())).convert(\"RGB\")\n\n\nclass Recortes:\n    \"\"\"Dataset de entrenamiento en recortes lado x lado a tamaño real (sin achicar la foto entera).\n    En las editadas, la mayoría de los recortes caen sobre la zona cambiada. La etiqueta es del RECORTE:\n    1 si contiene parte de la zona editada.\"\"\"\n\n    def __init__(self, filas: list[dict], lado: int = 512, semilla: int = 0, sobre_zona: float = .6):\n        self.filas, self.lado, self.semilla, self.sobre_zona = filas, lado, semilla, sobre_zona\n        self.epoca = 0\n\n    def __len__(self):\n        return len(self.filas)\n\n    def __getitem__(self, i):\n        f, L = self.filas[i], self.lado\n        rng = random.Random(hash((self.semilla, self.epoca, i)))\n        img = Image.open(f[\"ruta\"]).convert(\"RGB\")\n        if f[\"mascara\"]:\n            mask = Image.open(f[\"mascara\"]).convert(\"L\").resize(img.size, Image.NEAREST)\n        else:\n            mask = Image.new(\"L\", img.size, 255 if f[\"etiqueta\"] else 0)\n        img, mask = como_whatsapp(img, mask, rng)\n        if max(img.size) > MAX_LADO:                         # como_whatsapp ya deja <= 2048; por si acaso\n            img = preparar(img)\n            mask = mask.resize(img.size, Image.NEAREST)\n        if rng.random() < .5:\n            img, mask = img.transpose(Image.FLIP_LEFT_RIGHT), mask.transpose(Image.FLIP_LEFT_RIGHT)\n        w, h = img.size\n        m = np.asarray(mask) > 127\n        if f[\"etiqueta\"] and m.any() and not m.all() and rng.random() < self.sobre_zona:\n            ys, xs = np.nonzero(m)\n            k = rng.randrange(len(xs))\n            cx, cy = xs[k] + rng.randint(-L // 4, L // 4), ys[k] + rng.randint(-L // 4, L // 4)\n            x0, y0 = cx - L // 2, cy - L // 2\n        else:\n            x0, y0 = rng.randint(0, max(0, w - L)), rng.randint(0, max(0, h - L))\n        x0, y0 = min(max(0, x0), max(0, w - L)), min(max(0, y0), max(0, h - L))\n        tiles, _, _, _ = mosaico(img.crop((x0, y0, min(w, x0 + L), min(h, y0 + L))), L, L)\n        mm = np.zeros((L, L), np.float32)\n        mc = m[y0:y0 + L, x0:x0 + L]\n        mm[:mc.shape[0], :mc.shape[1]] = mc\n        return tiles[0], mm[None], np.float32(mm.mean() > .001)\n\n\nclass Conjunto:\n    \"\"\"Dataset compatible con torch.utils.data (sin depender de torch aquí).\"\"\"\n\n    def __init__(self, filas: list[dict], lado: int = 512, aumentar: bool = True, semilla: int = 0):\n        self.filas, self.lado, self.aumentar, self.semilla = filas, lado, aumentar, semilla\n        self.epoca = 0\n\n    def __len__(self):\n        return len(self.filas)\n\n    def __getitem__(self, i):\n        f = self.filas[i]\n        rng = random.Random(hash((self.semilla, self.epoca if self.aumentar else 0, i)))\n        img = Image.open(f[\"ruta\"]).convert(\"RGB\")\n        if f[\"mascara\"]:\n            mask = Image.open(f[\"mascara\"]).convert(\"L\").resize(img.size, Image.NEAREST)\n        else:\n            mask = Image.new(\"L\", img.size, 255 if (f[\"etiqueta\"] and not f[\"mascara\"]) else 0)\n        img, mask = como_whatsapp(img, mask, rng)\n        if self.aumentar and rng.random() < .5:\n            img, mask = img.transpose(Image.FLIP_LEFT_RIGHT), mask.transpose(Image.FLIP_LEFT_RIGHT)\n        m = (np.asarray(mask.resize((self.lado, self.lado), Image.NEAREST)) > 127).astype(np.float32)\n        return a_tensor(img, self.lado), m[None], np.float32(f[\"etiqueta\"])\n\n\ndef dividir(filas: list[dict], val: float = .15, semilla: int = 0) -> tuple[list[dict], list[dict]]:\n    \"\"\"Separa por imagen base (id COCO / nombre), para que la original y su edición no queden una en\n    entrenamiento y otra en validación (eso inflaría las métricas).\"\"\"\n    def base(f):\n        return re.match(r\"(\\d+|[^_.]+)\", Path(f[\"ruta\"]).name).group(1)\n    ids = sorted({base(f) for f in filas})\n    random.Random(semilla).shuffle(ids)\n    v = set(ids[:max(1, int(len(ids) * val))])\n    return [f for f in filas if base(f) not in v], [f for f in filas if base(f) in v]\n\n\nif __name__ == \"__main__\":\n    import argparse\n    ap = argparse.ArgumentParser(description=\"Crea el manifiesto CSV de entrenamiento.\")\n    ap.add_argument(\"--tgif\", type=Path, action=\"append\", default=[], help=\"carpeta raíz de TGIF/TGIF2\")\n    ap.add_argument(\"--propias\", type=Path, action=\"append\", default=[], help=\"carpeta con originales/ y editadas/\")\n    ap.add_argument(\"--salida\", type=Path, default=Path(\"manifiesto.csv\"))\n    a = ap.parse_args()\n    filas = [f for r in a.tgif for f in manifiesto_tgif(r)] + [f for r in a.propias for f in manifiesto_propias(r)]\n    guardar(filas, a.salida)\n    print(f\"{len(filas)} imágenes ({sum(f['etiqueta'] for f in filas)} editadas) -> {a.salida}\")\n",
 "modelo.py": "\"\"\"Red del detector: codificador timm + decodificador tipo FPN liviano.\n\nSalida: un mapa (logits, 1 canal, 1/4 de la resolución de entrada) de «zona editada con IA».\nEl puntaje de la imagen se calcula del mapa (promedio del 1 % de píxeles más altos), así la imagen y la zona\nsalen de lo mismo y el analista siempre ve DÓNDE está el indicio.\n\nCodificador por defecto: efficientnet_b0 (código Apache 2.0). Con --preentrenado se descargan pesos ImageNet\nde timm; revise sus condiciones antes de vender el modelo (ver training/README.md).\n\"\"\"\nfrom __future__ import annotations\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\n\nclass EvidexNet(nn.Module):\n    def __init__(self, encoder: str = \"efficientnet_b0\", preentrenado: bool = False, canales: int = 96):\n        super().__init__()\n        import timm\n        self.encoder = timm.create_model(encoder, features_only=True, pretrained=preentrenado,\n                                         out_indices=(1, 2, 3, 4))\n        chs = self.encoder.feature_info.channels()\n        self.lat = nn.ModuleList(nn.Conv2d(c, canales, 1) for c in chs)\n        self.suave = nn.Sequential(nn.Conv2d(canales, canales, 3, padding=1), nn.BatchNorm2d(canales),\n                                   nn.ReLU(inplace=True), nn.Conv2d(canales, 1, 1))\n\n    def forward(self, x):\n        feats = self.encoder(x)\n        y = self.lat[-1](feats[-1])\n        for f, lat in zip(reversed(feats[:-1]), reversed(self.lat[:-1])):\n            y = F.interpolate(y, size=f.shape[-2:], mode=\"bilinear\", align_corners=False) + lat(f)\n        return self.suave(y)                                  # (N,1,H/4,W/4) logits\n\n\ndef puntaje(logits: torch.Tensor, frac: float = .01) -> torch.Tensor:\n    \"\"\"Puntaje 0..1 por imagen: media del `frac` de píxeles más sospechosos.\"\"\"\n    p = torch.sigmoid(logits).flatten(1)\n    k = max(1, int(p.shape[1] * frac))\n    return p.topk(k, dim=1).values.mean(1)\n\n\nclass ConPuntaje(nn.Module):\n    \"\"\"Lo que se exporta a ONNX: devuelve (mapa de probabilidad, puntaje).\"\"\"\n\n    def __init__(self, red: EvidexNet):\n        super().__init__()\n        self.red = red\n\n    def forward(self, x):\n        logits = self.red(x)\n        return torch.sigmoid(logits), puntaje(logits)\n",
 "entrenar.py": "\"\"\"Entrena el detector de ediciones con IA y lo exporta para Evidex.\n\n  python training/entrenar.py --manifiesto manifiesto.csv --salida modelos/ --epocas 15 --preentrenado\n\nDeja en --salida:\n  evidex_ia.onnx         el modelo que carga Evidex (onnxruntime, sin torch en el servidor)\n  evidex_ia.safetensors  los pesos, para seguir entrenando (no se usa pickle: un pickle puede ejecutar código)\n  evidex_ia.json         ficha del modelo: conjuntos y licencias, umbral, métricas de validación, fecha\n\nEl umbral se elige en validación para que como máximo el --fpr de las ORIGINALES dé alerta (5 % por defecto).\nNecesita GPU para un entrenamiento real (Kaggle/Colab gratis sirven); en CPU solo para probar que corre.\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport sys\nimport time\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\n\nsys.path.insert(0, str(Path(__file__).resolve().parent))\nfrom datos import MAX_LADO, Recortes, degradar_fijo, dividir, leer, puntuar_mosaico  # noqa: E402\nfrom modelo import ConPuntaje, EvidexNet, puntaje  # noqa: E402\n\nFORMATO = 2          # 2 = mosaico a tamaño real (1 = foto entera achicada). Evidex revisa este número.\n\n\ndef perdida(logits, mask, etiqueta):\n    m = F.interpolate(mask, size=logits.shape[-2:], mode=\"nearest\")\n    bce = F.binary_cross_entropy_with_logits(logits, m)\n    p = torch.sigmoid(logits)\n    dice = 1 - (2 * (p * m).sum((1, 2, 3)) + 1) / (p.sum((1, 2, 3)) + m.sum((1, 2, 3)) + 1)\n    dice = (dice * etiqueta).sum() / etiqueta.sum().clamp(min=1)       # dice solo en editadas\n    s = puntaje(logits).clamp(1e-4, 1 - 1e-4)\n    img = F.binary_cross_entropy(s, etiqueta)\n    return bce + dice + img\n\n\n@torch.no_grad()\ndef evaluar(red, filas, disp, lado, paso):\n    \"\"\"Cada foto de validación entera, pasada por WhatsApp (1600 px, JPEG 70) y analizada en mosaico,\n    igual que la analizará Evidex. Así el umbral queda calibrado para fotos completas.\"\"\"\n    from PIL import Image\n    red.eval()\n\n    def correr(t):\n        logits = red(torch.from_numpy(t).to(disp))\n        return torch.sigmoid(logits).cpu().numpy(), puntaje(logits).cpu().numpy()\n\n    ps, ys, ious = [], [], []\n    for f in filas:\n        img = degradar_fijo(Image.open(f[\"ruta\"]))\n        s, mapa, esc = puntuar_mosaico(correr, img, lado, paso)\n        ps.append(s)\n        ys.append(f[\"etiqueta\"])\n        if f[\"etiqueta\"]:\n            if f[\"mascara\"]:\n                m = Image.open(f[\"mascara\"]).convert(\"L\").resize((mapa.shape[1], mapa.shape[0]), Image.NEAREST)\n                m = np.asarray(m) > 127\n            else:\n                m = np.ones(mapa.shape, bool)\n            a = mapa > .5\n            union = (a | m).sum()\n            ious.append(float((a & m).sum() / union) if union else 1.0)\n    return np.array(ps), np.array(ys), float(np.mean(ious)) if ious else None\n\n\ndef umbral_para(ps, ys, fpr):\n    orig = np.sort(ps[ys == 0])\n    if len(orig) == 0:\n        return .5\n    k = int(np.floor(len(orig) * (1 - fpr)))\n    return float(orig[min(k, len(orig) - 1)]) + 1e-6\n\n\ndef metricas(ps, ys, umbral):\n    ed, ori = ps[ys == 1], ps[ys == 0]\n    return {\"deteccion_editadas\": float((ed >= umbral).mean()) if len(ed) else None,\n            \"falsas_alarmas_originales\": float((ori >= umbral).mean()) if len(ori) else None,\n            \"n_editadas\": int(len(ed)), \"n_originales\": int(len(ori))}\n\n\ndef exportar(red, lado, destino: Path):\n    red = red.cpu().eval()\n    x = torch.zeros(1, 3, lado, lado)\n    torch.onnx.export(ConPuntaje(red), (x,), str(destino), input_names=[\"imagen\"],\n                      output_names=[\"mapa\", \"puntaje\"], opset_version=17, dynamo=False,\n                      dynamic_axes={\"imagen\": {0: \"n\"}, \"mapa\": {0: \"n\"}, \"puntaje\": {0: \"n\"}})\n\n\ndef main(argv=None):\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--manifiesto\", type=Path, required=True)\n    ap.add_argument(\"--salida\", type=Path, default=Path(\"modelos\"))\n    ap.add_argument(\"--encoder\", default=\"efficientnet_b0\")\n    ap.add_argument(\"--preentrenado\", action=\"store_true\", help=\"partir de pesos ImageNet de timm\")\n    ap.add_argument(\"--lado\", type=int, default=512)\n    ap.add_argument(\"--epocas\", type=int, default=15)\n    ap.add_argument(\"--lote\", type=int, default=12)\n    ap.add_argument(\"--lr\", type=float, default=3e-4)\n    ap.add_argument(\"--fpr\", type=float, default=.05, help=\"falsas alarmas máximas en originales (validación)\")\n    ap.add_argument(\"--limite\", type=int, default=0, help=\"usar solo N imágenes (prueba rápida)\")\n    ap.add_argument(\"--trabajadores\", type=int, default=2)\n    ap.add_argument(\"--max-horas\", type=float, default=0,\n                    help=\"parar antes de este tiempo y exportar lo mejor (Kaggle corta a las 12 h)\")\n    ap.add_argument(\"--val-max\", type=int, default=800, help=\"fotos de validación por época (las más lentas)\")\n    ap.add_argument(\"--continuar\", type=Path, help=\"evidex_ia.safetensors de una vuelta anterior (mismo encoder)\")\n    a = ap.parse_args(argv)\n\n    filas = leer(a.manifiesto)\n    if a.limite:\n        filas = filas[:a.limite // 2] + filas[-(a.limite - a.limite // 2):]\n    entren, val = dividir(filas)\n    disp = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    red = EvidexNet(a.encoder, a.preentrenado and not a.continuar)\n    if a.continuar:\n        from safetensors.torch import load_file\n        red.load_state_dict(load_file(str(a.continuar)))\n        print(f\"Continuando desde {a.continuar}\", flush=True)\n    red = red.to(disp)\n    inicio, ultima = time.time(), 0.0\n    opt = torch.optim.AdamW(red.parameters(), lr=a.lr, weight_decay=1e-4)\n    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, a.epocas))\n    paso = a.lado * 3 // 4                       # los pedazos se traslapan un cuarto\n    ds = Recortes(entren, a.lado)\n    import random as _r\n    val = sorted(val, key=lambda f: f[\"ruta\"])\n    _r.Random(1).shuffle(val)\n    val = [f for f in val if f[\"etiqueta\"] == 0][:a.val_max // 2] + [f for f in val if f[\"etiqueta\"]][:a.val_max // 2]\n    mejor, hist = None, []\n    a.salida.mkdir(parents=True, exist_ok=True)\n    for ep in range(a.epocas):\n        if a.max_horas and time.time() - inicio + ultima * 1.1 > a.max_horas * 3600:\n            print(f\"Se acaba el tiempo ({a.max_horas} h): se para en la época {ep} y se exporta lo mejor.\", flush=True)\n            break\n        ds.epoca = ep\n        red.train()\n        t, tot = time.time(), 0.0\n        for x, m, y in DataLoader(ds, batch_size=a.lote, shuffle=True, num_workers=a.trabajadores, drop_last=True):\n            loss = perdida(red(x.to(disp)), m.to(disp), y.to(disp))\n            opt.zero_grad()\n            loss.backward()\n            opt.step()\n            tot += loss.item()\n        sched.step()\n        ps, ys, iou = evaluar(red, val, disp, a.lado, paso)\n        u = umbral_para(ps, ys, a.fpr)\n        met = {**metricas(ps, ys, u), \"iou_zona\": iou, \"umbral\": u}\n        hist.append({\"epoca\": ep + 1, \"perdida\": tot, **met})\n        ultima = time.time() - t\n        print(json.dumps(hist[-1], ensure_ascii=False), f\"({ultima:.0f}s)\", flush=True)\n        if mejor is None or (met[\"deteccion_editadas\"] or 0) > (mejor[\"deteccion_editadas\"] or 0):\n            mejor = met\n            from safetensors.torch import save_file\n            save_file({k: v.contiguous().cpu() for k, v in red.state_dict().items()},\n                      str(a.salida / \"evidex_ia.safetensors\"))\n\n    from safetensors.torch import load_file\n    red.load_state_dict(load_file(str(a.salida / \"evidex_ia.safetensors\")))\n    exportar(red, a.lado, a.salida / \"evidex_ia.onnx\")\n    conjuntos = Counter((f[\"conjunto\"].split(\"/\")[0], f[\"licencia\"]) for f in filas)\n    ficha = {\n        \"formato\": FORMATO, \"nombre\": \"Evidex IA (zonas editadas con IA)\", \"encoder\": a.encoder,\n        \"preentrenado\": \"ImageNet (timm)\" if a.preentrenado else None,\n        \"continuado_desde\": str(a.continuar) if a.continuar else None, \"lado\": a.lado,\n        \"modo\": \"mosaico\", \"paso\": paso, \"max_lado\": MAX_LADO,\n        \"normalizacion\": \"imagenet\", \"umbral\": mejor[\"umbral\"], \"fpr_objetivo\": a.fpr,\n        \"validacion\": mejor, \"historial\": hist,\n        \"conjuntos\": [{\"nombre\": n, \"licencia\": l, \"imagenes\": c} for (n, l), c in sorted(conjuntos.items())],\n        \"aumentos\": \"como WhatsApp: reducción 640-2048 px, JPEG 50-92 (a veces doble), recorte leve, espejo; \"\n                    \"recortes a tamaño real (60 % sobre la zona editada)\",\n        \"fecha\": datetime.now(timezone.utc).isoformat(timespec=\"seconds\"),\n        \"sha256_onnx\": hashlib.sha256((a.salida / \"evidex_ia.onnx\").read_bytes()).hexdigest(),\n        \"advertencia\": \"Indicio para revisar, no prueba. Validar con fotos reales de siniestros antes de usar.\",\n    }\n    (a.salida / \"evidex_ia.json\").write_text(json.dumps(ficha, indent=2, ensure_ascii=False), encoding=\"utf-8\")\n    print(f\"Listo: {a.salida / 'evidex_ia.onnx'}  umbral={mejor['umbral']:.3f}  validación={mejor}\")\n\n\nif __name__ == \"__main__\":\n    main()\n",
 "descargar_tgif.py": "\"\"\"Descarga una parte de TGIF (CC BY 4.0) sin bajar los archivos completos (algunos pesan 14 GB).\n\nLee el .tar.gz a medida que llega y se detiene al tener N imágenes de cada carpeta.\n  python training/descargar_tgif.py --destino datos/tgif --n 3000 --particion training\n  python training/descargar_tgif.py --destino datos/tgif --n 500  --particion validation\n\nOrigen: https://github.com/IDLabMedia/tgif-dataset (enlaces públicos de Nextcloud de IDLab, imec).\nCite el artículo de TGIF y respete CC BY 4.0 (atribución). No se bajan las variantes FLUX (licencia no comercial).\n\"\"\"\nfrom __future__ import annotations\n\nimport argparse\nimport tarfile\nimport urllib.request\nfrom pathlib import Path\n\nBASE = \"https://cloud.ilabt.imec.be/public.php/dav/files/xEeAzrY7ES9KA8o\"\nCARPETAS = (\"orig\", \"sd2-sp\", \"sd2-fr\", \"sdxl-fr\", \"ps-sp\")\n\n\ndef bajar(carpeta: str, particion: str, destino: Path, n: int) -> int:\n    url = f\"{BASE}/{carpeta}/{carpeta}_{particion}.tar.gz\"\n    req = urllib.request.Request(url, headers={\"X-Requested-With\": \"XMLHttpRequest\"})\n    k = 0\n    with urllib.request.urlopen(req, timeout=120) as r, tarfile.open(fileobj=r, mode=\"r|gz\") as t:\n        for m in t:\n            if not m.isfile():\n                continue\n            t.extract(m, destino / carpeta, filter=\"data\")\n            k += 1\n            if n and k >= n:\n                break\n    return k\n\n\ndef main():\n    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)\n    ap.add_argument(\"--destino\", type=Path, required=True)\n    ap.add_argument(\"--n\", type=int, default=2000, help=\"imágenes por carpeta (0 = todas)\")\n    ap.add_argument(\"--particion\", choices=(\"training\", \"validation\", \"testing\"), default=\"training\")\n    ap.add_argument(\"--carpetas\", default=\",\".join(CARPETAS))\n    a = ap.parse_args()\n    for c in a.carpetas.split(\",\"):\n        print(c, bajar(c, a.particion, a.destino, a.n), flush=True)\n    print(\"masks\", bajar(\"masks\", a.particion, a.destino, 0))      # las máscaras pesan poco: todas\n\n\nif __name__ == \"__main__\":\n    main()\n"
}
for nombre, contenido in ARCHIVOS.items():
    Path("training", nombre).write_text(contenido, encoding="utf-8")
print("Archivos:", ", ".join(ARCHIVOS))

In [ ]:
# 4. Bajar una parte de TGIF (lee el .tar.gz mientras llega; no baja los 14 GB completos)
sh(f"{sys.executable} training/descargar_tgif.py --destino {DATOS} --n {N_ORIG} --particion training --carpetas orig")
sh(f"{sys.executable} training/descargar_tgif.py --destino {DATOS} --n {N_POR_CARPETA} --particion training --carpetas {CARPETAS}")

In [ ]:
# 5. Manifiesto: qué imágenes se usan y con qué licencia (se niega si hay alguna no comercial)
import glob
propias = [os.path.dirname(p) for p in glob.glob("/kaggle/input/*/originales") + glob.glob("/kaggle/input/*/*/originales")]
extra = " ".join(f"--propias {p}" for p in propias)
print("Fotos propias:", propias or "ninguna")
sh(f"{sys.executable} training/datos.py --tgif {DATOS} {extra} --salida manifiesto.csv")

In [ ]:
# 6. Entrenar (aquí se van las horas; cada época imprime cuántas editadas detecta en validación)
# Si agregó con «+ Add Input» el modelo de la vuelta anterior (evidex_modelo.zip subido como Dataset),
# se parte desde él en vez de empezar de cero: aprende más en el mismo tiempo.
previo = glob.glob("/kaggle/input/**/evidex_ia.safetensors", recursive=True)
pre = f"--continuar {previo[0]}" if previo else ("--preentrenado" if PREENTRENADO else "")
print("Modelo anterior:", previo[0] if previo else "ninguno (se parte de cero)")
sh(f"{sys.executable} training/entrenar.py --manifiesto manifiesto.csv --salida {SALIDA} "
   f"--epocas {EPOCAS} --lado {LADO} --lote {LOTE} --trabajadores 4 --max-horas {MAX_HORAS} {pre}")

In [ ]:
# 7. Medir con fotos que el modelo nunca vio (partición «testing»), pasadas por WhatsApp
import io, json, numpy as np, onnxruntime as ort
from PIL import Image
sys.path.insert(0, "training")
from datos import degradar_fijo, preparar, puntuar_mosaico
PRUEBA = DATOS + "_prueba"
sh(f"{sys.executable} training/descargar_tgif.py --destino {PRUEBA} --n {N_PRUEBA} --particion testing --carpetas orig,sd2-sp,sd2-fr,sdxl-fr,ps-sp")
ficha = json.load(open(f"{SALIDA}/evidex_ia.json"))
sess = ort.InferenceSession(f"{SALIDA}/evidex_ia.onnx", providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
correr = lambda t: sess.run(["mapa", "puntaje"], {"imagen": t})
def puntajes(carpeta):
    # igual que Evidex: foto pasada por WhatsApp (1600 px, JPEG 70) y analizada en mosaico a tamaño real
    out = []
    for p in sorted(glob.glob(f"{PRUEBA}/{carpeta}/**/*.png", recursive=True)):
        img = preparar(degradar_fijo(Image.open(p)), ficha["max_lado"])
        out.append(puntuar_mosaico(correr, img, ficha["lado"], ficha["paso"])[0])
    return np.array(out)
u = ficha["umbral"]
NOMBRES = {"orig": "falsas_alarmas_originales", "sd2-sp": "detectadas_sd2_zona", "sd2-fr": "detectadas_sd2_regenerada",
           "sdxl-fr": "detectadas_sdxl_regenerada", "ps-sp": "detectadas_photoshop_zona"}
res = {c: puntajes(c) for c in NOMBRES}
medicion = {"umbral": u, **{NOMBRES[c]: f"{(v >= u).sum()} de {len(v)}" for c, v in res.items() if len(v)},
            "vuelta_1 (foto achicada)": "falsas 4 de 300, sd2 zona 69 de 300, sdxl regenerada 215 de 300",
            "linea_base_evidex_hoy": "2-7 de cada 100 editadas pasadas por WhatsApp"}
ficha["prueba_testing_whatsapp"] = medicion
json.dump(ficha, open(f"{SALIDA}/evidex_ia.json", "w"), indent=2, ensure_ascii=False)   # el sha256 es del .onnx: sigue válido
print(json.dumps(medicion, indent=2, ensure_ascii=False))

In [ ]:
# 8. Empaquetar para descargar (pestaña Output → evidex_modelo.zip)
shutil.make_archive(os.path.join(os.path.dirname(SALIDA) or ".", "evidex_modelo"), "zip", SALIDA)
print("Listo. Descargue evidex_modelo.zip y copie su contenido a veritas\\modelos\\ en su PC.")
print(sorted(os.listdir(SALIDA)))